# Abu Dhabi Real Estate (ADREC) Transaction Data Processing Pipeline

This notebook executes the end-to-end cleaning, standardization, geocoding enrichment, and Non-RERA indexing workflow for Abu Dhabi property transaction data.

### Input Files Required:
1. **Raw Transaction CSV** (`AbuDhabi_RawData.csv`): Raw transaction records from ADREC.
2. **Project Coordinates CSV** (`Abudhabi_project_coordinates.csv`): Master reference for project coordinates and existing Non-RERA indexes.
3. **Location Coordinates CSV** (`Abudhabi_location_coordinates.csv`): Master reference for location IDs and coordinates.

### Pipeline Stages:
- **Step 2**: Load raw CSV data (dtype=str, utf-8-sig)
- **Step 3**: Drop duplicate transactions
- **Step 4**: Parse transaction dates and extract fiscal quarters
- **Step 5**: Standardize property type taxonomy (Flat, Villa, Office, Shop, Plot, Others)
- **Step 6**: Classify project types and asset classes
- **Step 7**: Normalize unit configurations (1Bhk, 2Bhk, etc.) and assign metadata
- **Step 8**: Enrich location coordinates and location IDs (with Jabal Hafeet fallback)
- **Step 9**: Match against master project coordinates reference
- **Step 10**: Geocode missing commercial projects & apply fallbacks
- **Step 11**: Assign sequential Non-RERA indexes (`internal_index_id` starting from `nr468`)
- **Step 12**: Align to 68-column DB schema and apply Title Case formatting
- **Step 13**: Export database-ready CSV and Excel outputs

In [ ]:
import os
import sys
import re
import numpy as np
import pandas as pd
import requests

print("Pandas version:", pd.__version__)

### 1. Configure Input & Output File Paths
Specify the paths to the three required input files:

In [ ]:
# ==============================================================================
# INPUT FILE PATH CONFIGURATION
# ==============================================================================

# 1. Raw transaction data from ADREC
RAW_DATA_PATH = "AbuDhabi_RawData.csv"

# 2. Master project coordinates and internal index reference table
PROJECT_COORDS_PATH = "Abudhabi_project_coordinates.csv"

# 3. Master location coordinates and location ID reference table
LOCATION_COORDS_PATH = "Abudhabi_location_coordinates.csv"

# Output file destinations
OUTPUT_CSV_PATH = "AbuDhabi_Processed_Output.csv"
OUTPUT_XLSX_PATH = "AbuDhabi_Processed_Output.xlsx"

print("Configured Input Files:")
print("  - Raw Transaction CSV       :", RAW_DATA_PATH, f"(Found: {os.path.exists(RAW_DATA_PATH)})")
print("  - Master Project Coordinates:", PROJECT_COORDS_PATH, f"(Found: {os.path.exists(PROJECT_COORDS_PATH)})")
print("  - Master Location Reference :", LOCATION_COORDS_PATH, f"(Found: {os.path.exists(LOCATION_COORDS_PATH)})")

### 2. Load Raw Transaction CSV (Step 2)
Read the raw transaction data with string dtypes to prevent data truncation or precision loss.

In [ ]:
# Load raw dataset with utf-8-sig encoding
df = pd.read_csv(RAW_DATA_PATH, dtype=str, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print(f"Loaded raw dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

### 3. Drop Duplicate Transactions (Step 3)
Identify and remove exact duplicate transaction records.

In [ ]:
before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
dropped_count = before_dedup - len(df)

print(f"Dropped {dropped_count:,} duplicate records.")
print(f"Remaining unique records: {len(df):,}")

### 4. Parse Dates & Extract Fiscal Quarters (Step 4)
Standardize dates to `DD-MM-YYYY` strings, extract `year` (`Int64`), and compute fiscal `quarter` (`QX-YYYY`).

In [ ]:
date_series = pd.to_datetime(df["Sale Application Date"], format="%Y-%m-%d", errors="coerce")
fallback_formats = [
    "%d-%m-%Y", "%Y/%m/%d", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S", "%d-%m-%Y %H:%M:%S"
]
for fmt in fallback_formats:
    if date_series.isna().any():
        date_series = date_series.fillna(pd.to_datetime(df["Sale Application Date"], format=fmt, errors="coerce"))
if date_series.isna().any():
    date_series = date_series.fillna(pd.to_datetime(df["Sale Application Date"], dayfirst=True, errors="coerce"))

df["transaction_date"] = date_series
df["date_of_agreement_execution"] = date_series
df["year"] = df["transaction_date"].dt.year.astype("Int64")
df["quarter"] = "Q" + df["transaction_date"].dt.quarter.astype("Int64").astype(str) + "-" + df["year"].astype(str)
df.loc[df["transaction_date"].isna(), "quarter"] = None

# Format to standard DD-MM-YYYY strings
df["transaction_date"] = df["transaction_date"].dt.strftime("%d-%m-%Y")
df["date_of_agreement_execution"] = df["date_of_agreement_execution"].dt.strftime("%d-%m-%Y")

print(f"Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")
print("Quarter counts:")
print(df["quarter"].value_counts(dropna=False))

### 5. Property Type Mapping (Step 5)
Standardize 40+ raw property types into standard categories: `Flat`, `Villa`, `Office`, `Shop`, `Plot`, and `Others`.

In [ ]:
PROPERTY_TYPE_MAP = {
    "apartment": "Flat",
    "duplex": "Flat",
    "residential complex": "Flat",
    "residential complex (staff accommodation)": "Flat",
    "villa": "Villa",
    "townhouse / attached villa": "Villa",
    "palace": "Villa",
    "office": "Office",
    "office complex": "Office",
    "retail": "Shop",
    "mall / market / retail center": "Shop",
    "other commercial plot": "Plot",
    "other industrial plot": "Plot",
    "other mixed-use plot": "Plot",
    "other plot": "Plot",
    "other residential plot": "Plot",
    "other temporary plot": "Plot",
    "plot for clinic / hospital": "Plot",
    "plot for cooling utility": "Plot",
    "plot for farm": "Plot",
    "plot for hotel / resort / serviced apartments": "Plot",
    "plot for mall / market / retail center": "Plot",
    "plot for office complex": "Plot",
    "plot for palace": "Plot",
    "plot for park": "Plot",
    "plot for parking": "Plot",
    "plot for petrol station": "Plot",
    "plot for residential complex": "Plot",
    "plot for residential complex (staff accommodation)": "Plot",
    "plot for school": "Plot",
    "plot for townhouse / attached villa": "Plot",
    "plot for villa": "Plot",
    "clinic / hospital": "Others",
    "cooling utility": "Others",
    "factory": "Others",
    "farm": "Others",
    "hotel / resort / serviced apartments": "Others",
    "mosque": "Others",
    "other": "Others",
    "penthouse": "Others",
    "petrol station": "Others",
    "school": "Others",
    "workshop": "Others"
}

df["property_type_raw"] = df["Property Type"].fillna("").astype(str).str.strip().str.lower()
df["property_type"] = df["property_type_raw"].map(PROPERTY_TYPE_MAP).fillna("Others")

print("Property type distribution:")
print(df["property_type"].value_counts())

### 6. Project Type & Asset Class Mapping (Step 6)
Classify raw asset classes into standardized project types (`Residential`, `Commercial`, `Industrial`, `Infrastructure`, `Agricultural`, `Educational`, `Healthcare`, `Recreational`, `Religious`, `Others`).

In [ ]:
PROJECT_TYPE_MAP = {
    "agricultural": "Agricultural",
    "commercial": "Commercial",
    "educational": "Educational",
    "healthcare": "Healthcare",
    "industrial & storage": "Industrial",
    "infrastructural": "Infrastructure",
    "other": "Others",
    "recreational": "Recreational",
    "religious": "Religious",
    "residential": "Residential"
}

df["project_type"] = (
    df["Asset Class"].fillna("").astype(str).str.strip().str.lower().map(PROJECT_TYPE_MAP).fillna("Others")
)

print("Project type distribution:")
print(df["project_type"].value_counts())

### 7. Unit Configuration Normalization & Default Metadata (Step 7)
Normalize unit layouts (`1Bhk`, `2Bhk`, `Shop`, `Office`, etc.) and set default static metadata for Abu Dhabi.

In [ ]:
NORMALIZED_UNIT_CONFIG_MAP = {
    "1 bed": "1Bhk",
    "2 beds": "2Bhk",
    "3 beds": "3Bhk",
    "4 beds": ">3Bhk",
    "5 beds": ">3Bhk",
    "5+ beds": ">3Bhk",
    "6+ beds": ">3Bhk",
    "anchor store": "Shop",
    "large (500 to 1000 sqm)": "Office",
    "line store": "Shop",
    "maxi (over 1000 sqm)": "Office",
    "medium (50 to 500 sqm)": "Office",
    "small (up to 50 sqm)": "Office",
    "studio": "Flat",
    "unclassified": "Others"
}

df["unit_configuration"] = df["Property Layout"].fillna("").astype(str).str.strip().str.lower()
df["property_layout"] = df["Property Layout"]
df["normalized_unit_configuration"] = df["unit_configuration"].map(NORMALIZED_UNIT_CONFIG_MAP).fillna("Others")

# Direct source mappings
df["project_name"] = df["Project Name"]
df["location_name"] = df["District"]
df["agreement_price"] = pd.to_numeric(df["Property Sale Price (AED)"], errors="coerce")
df["net_carpet_area_sq_m"] = pd.to_numeric(df["Property Sold Area (SQM)"], errors="coerce")
df["project_stage"] = df["Sale Application Type"]
df["sale_type"] = df.get("Sale Sequence", None)
df["sub_locality"] = df.get("Community", None)

# Columns explicitly kept null by design
df["micro_market"] = None
df["property_description"] = None

# Static Abu Dhabi defaults
df["city_id"] = 1
df["city_name"] = "Abu Dhabi"
df["state_name"] = "Abu Dhabi"
df["country_name"] = "UAE"
df["transaction_category"] = "Sale"
df["data_source"] = "Adrec"
df["data_type"] = "Registered Document"
df["source_accessibility"] = "Download"
df["source_accessibility_way"] = "Easy"
df["is_llm_processed"] = "No"
df["is_manual_processed"] = "No"

# Placeholder columns
default_blank_cols = [
    "registered_document_village_name", "sub_registrar_office_code", "sub_registrar_office_name",
    "document_number", "guideline_value", "floor_number", "unit_number", "balcony_sq_m",
    "terrace_sq_m", "seller_name", "buyer_name", "internal_document_number", "micr_number",
    "bank_type", "party_code", "stamp_duty_paid", "registration_fee", "project_latitude",
    "project_longitude", "location_latitude", "location_longitude", "location_id",
    "internal_index_id", "buyer_pincode", "buyer_locality", "buyer_district", "buyer_state", "tower_name",
    "is_duplicate", "pincode", "parking_count", "facing_direction", "view_type", "furnishing_status",
    "condition_status", "sourcing_cost", "sourcing_time"
]
for col in default_blank_cols:
    if col not in df.columns:
        df[col] = None

print("Normalized unit configuration counts:")
print(df["normalized_unit_configuration"].value_counts())

### 8. Master Location Coordinates & Location ID Lookup (Step 8)
Enrich district-level coordinates (`location_latitude`, `location_longitude`) and `location_id` from `Abudhabi_location_coordinates.csv`, including fallback for Jabal Hafeet.

In [ ]:
def norm(s):
    if isinstance(s, pd.Series):
        return s.astype(str).str.strip().str.lower()
    return str(s).strip().lower()

df_loc = pd.read_csv(LOCATION_COORDS_PATH)
df_loc["norm_loc"] = norm(df_loc["location_name"])

loc_id_map = df_loc.drop_duplicates("norm_loc").set_index("norm_loc")["location_id"].to_dict()
loc_lat_map = df_loc.drop_duplicates("norm_loc").set_index("norm_loc")["location_latitude"].to_dict()
loc_lng_map = df_loc.drop_duplicates("norm_loc").set_index("norm_loc")["location_longitude"].to_dict()

df["norm_loc"] = norm(df["location_name"])
df["location_id"] = df["norm_loc"].map(loc_id_map)
df["location_latitude"] = pd.to_numeric(df["norm_loc"].map(loc_lat_map), errors="coerce")
df["location_longitude"] = pd.to_numeric(df["norm_loc"].map(loc_lng_map), errors="coerce")

# Known fallback for Jabal Hafeet
df.loc[df["norm_loc"] == "jabal hafeet", ["location_latitude", "location_longitude"]] = [24.0716702, 55.7985914]

matched_loc_count = int(df["location_latitude"].notna().sum())
print(f"Location coordinates mapped: {matched_loc_count:,}/{len(df):,} rows")
print(f"Distinct locations mapped   : {df['location_id'].nunique()}")

### 9. Master Project Coordinates & Internal Index ID Matching (Step 9)
Match projects against `Abudhabi_project_coordinates.csv` using composite key `(location_name, project_name, city_name)`. Unnamed or missing project names become `'Private'`.

In [ ]:
def clean_nr_index(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = re.sub(r"__.*$", "", x)
    m = re.match(r"(?i)^nr(\d+)", x)
    return f"nr{m.group(1)}" if m else x

df_proj = pd.read_csv(PROJECT_COORDS_PATH)

# Enrich project master with location_name via location_id
df_proj_ref = df_proj.merge(df_loc[["location_id", "location_name"]], on="location_id", how="left")
df_proj_ref["city_name"] = "Abu Dhabi"
df_proj_ref["norm_proj"] = norm(df_proj_ref["project_name"])
df_proj_ref["norm_loc"] = norm(df_proj_ref["location_name"])
df_proj_ref["norm_city"] = norm(df_proj_ref["city_name"])
df_proj_ref["clean_index"] = df_proj_ref["internal_index_id"].map(clean_nr_index)

master_lat_map = {}
master_lng_map = {}
master_index_map = {}

for _, r in df_proj_ref.iterrows():
    k = (r["norm_loc"], r["norm_proj"], r["norm_city"])
    master_lat_map[k] = r["project_latitude"]
    master_lng_map[k] = r["project_longitude"]
    master_index_map[k] = r["clean_index"]

# Normalize project_name: ensure empty, whitespace, or missing project names become 'Private'
df["project_name"] = df["project_name"].fillna("Private")
blank_mask = df["project_name"].astype(str).str.strip().isin(["", "nan", "<NA>", "None"])
df.loc[blank_mask, "project_name"] = "Private"

df["norm_proj"] = norm(df["project_name"])
df["norm_city"] = norm(df["city_name"])

# Populate matched master coordinates and index
for i in df.index:
    k = (df.at[i, "norm_loc"], df.at[i, "norm_proj"], df.at[i, "norm_city"])
    if k in master_lat_map:
        df.at[i, "project_latitude"] = master_lat_map[k]
        df.at[i, "project_longitude"] = master_lng_map[k]
        df.at[i, "internal_index_id"] = master_index_map[k]

master_matched_rows = int(df["project_latitude"].notna().sum())
print(f"Master project coordinates matched: {master_matched_rows:,}/{len(df):,} rows")

### 10. Commercial Project Geocoding & Fallbacks (Step 10)
For named commercial projects missing coordinates, unresolved commercial projects fall back to district location coordinates. Private projects intentionally remain blank (no coordinates).

In [ ]:
df["project_latitude"] = pd.to_numeric(df["project_latitude"], errors="coerce")
df["project_longitude"] = pd.to_numeric(df["project_longitude"], errors="coerce")

# For commercial projects not in master, fallback to district location coordinates
comm_missing_mask = (df["norm_proj"] != "private") & df["project_latitude"].isna()
df.loc[comm_missing_mask, "project_latitude"] = df.loc[comm_missing_mask, "location_latitude"]
df.loc[comm_missing_mask, "project_longitude"] = df.loc[comm_missing_mask, "location_longitude"]

valid_coords = int(df["project_latitude"].notna().sum())
print(f"Total rows with project coordinates : {valid_coords:,}/{len(df):,}")
print(f"Private rows (coordinates kept blank): {(df['norm_proj'] == 'private').sum():,}")

### 11. Assign Non-RERA Index (internal_index_id) (Step 11)
Assign sequential Non-RERA indexes (`internal_index_id`) starting from `nr468` (highest master index `nr467` + 1). Both commercial and `'Private'` projects sharing the same name and location receive the identical `internal_index_id`.

In [ ]:
master_nr_numbers = (
    df_proj["internal_index_id"]
    .dropna()
    .str.extract(r"(?i)^nr(\d+)")[0]
    .dropna()
    .astype(int)
)
max_nr = int(master_nr_numbers.max()) if not master_nr_numbers.empty else 467
next_num = max_nr + 1

print(f"Master table highest NR = nr{max_nr}")
print(f"New projects will start from = nr{next_num}")

existing_indexes = set(master_index_map.values())
file_mapping = {}
new_nr_count = 0

for i in df.index:
    if pd.isna(df.at[i, "internal_index_id"]):
        p_name = df.at[i, "project_name"]
        l_name = df.at[i, "location_name"]
        p_norm = df.at[i, "norm_proj"]

        if pd.isna(l_name) or norm(l_name) in ["", "nan"]:
            l_name = "Unknown Location"

        # Shared composite key: [ location + project + city ]
        key = (norm(l_name), p_norm, norm(df.at[i, "city_name"]))
        if key in file_mapping:
            df.at[i, "internal_index_id"] = file_mapping[key]
        else:
            while f"nr{next_num}" in existing_indexes:
                next_num += 1
            new_index = f"nr{next_num}"
            df.at[i, "internal_index_id"] = new_index
            file_mapping[key] = new_index
            existing_indexes.add(new_index)
            next_num += 1
            new_nr_count += 1

df["internal_index_id"] = df["internal_index_id"].map(clean_nr_index)
total_with_internal_index = int(df["internal_index_id"].notna().sum())

print(f"New NR indexes assigned       : {new_nr_count} distinct projects (ending at nr{next_num - 1})")
print(f"Total rows with internal_index_id: {total_with_internal_index:,}/{len(df):,}")

### 12. DB Schema Alignment & Title Case (Step 12)
Reorder columns strictly according to the 68-column DB1 schema. Place `internal_index_id` at index 0, purge `project_id`, ensure `property_description` and `micro_market` remain null, and format string columns in Title Case.

In [ ]:
final_schema_order = [
    "internal_index_id", "project_name", "location_id", "location_name",
    "registered_document_village_name", "agreement_price", "transaction_date",
    "net_carpet_area_sq_m", "sub_registrar_office_code", "sub_registrar_office_name",
    "document_number", "transaction_type", "city_id", "city_name",
    "guideline_value", "property_description", "floor_number", "unit_number",
    "property_type_raw", "property_type", "year", "balcony_sq_m", "terrace_sq_m",
    "seller_name", "buyer_name", "transaction_category", "internal_document_number",
    "micr_number", "bank_type", "party_code", "date_of_agreement_execution",
    "stamp_duty_paid", "registration_fee", "project_latitude", "project_longitude",
    "location_latitude", "location_longitude", "quarter", "unit_configuration",
    "property_layout", "buyer_pincode", "buyer_locality", "buyer_district",
    "buyer_state", "is_llm_processed", "is_manual_processed", "tower_name",
    "is_duplicate", "sale_type", "project_type", "country_name", "state_name",
    "micro_market", "sub_locality", "pincode", "parking_count", "facing_direction",
    "view_type", "furnishing_status", "condition_status", "source_accessibility",
    "source_accessibility_way", "sourcing_cost", "sourcing_time", "data_type",
    "data_source", "normalized_unit_configuration", "project_stage"
]

# Ensure project_id is purged
if "project_id" in df.columns:
    df = df.drop(columns=["project_id"])

# Ensure all schema columns exist
for col in final_schema_order:
    if col not in df.columns:
        df[col] = None

# Reorder
df = df[final_schema_order]

# Apply Title Case to all object/text columns except internal_index_id
for col in df.select_dtypes(include=["object", "string"]).columns:
    if col != "internal_index_id":
        df[col] = df[col].apply(
            lambda x: x.title() if isinstance(x, str) and x.lower() not in ["none", "nan", "<na>", ""] else x
        )

# Guarantee property_description and micro_market remain strictly None
df["property_description"] = None
df["micro_market"] = None

print(f"Final aligned dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"First 5 columns: {list(df.columns[:5])}")
df.head(2)

### 13. Export Final Clean Datasets (Step 13)
Save the standardized dataset to CSV (`utf-8-sig`) and Excel (`.xlsx`) files.

In [ ]:
# Save CSV
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"[OK] Exported CSV: {OUTPUT_CSV_PATH}")

# Save Excel
try:
    df.to_excel(OUTPUT_XLSX_PATH, index=False)
    print(f"[OK] Exported Excel: {OUTPUT_XLSX_PATH}")
except Exception as e:
    print(f"[WARNING] Could not write Excel file ({e}). CSV was successfully exported.")

print("\n" + "=" * 65)
print("PIPELINE EXECUTION COMPLETED SUCCESSFULLY!")
print(f"   • Total records processed      : {len(df):,}")
print(f"   • Records with internal_index_id: {df['internal_index_id'].notna().sum():,}")
print(f"   • Private records processed    : {(df['project_name'].str.lower() == 'private').sum():,}")
print("=" * 65)